# Tutorial: CNNApprox on Real Impedance Data

Learning goals:
- `ConvolutionalInverseApproximator` на частотном окне реального датасета.
- Можно интерпретировать графики `training_loss`, `gmres_iterations_vs_frequency`, `preconditioner_consistency_vs_frequency` и `inverse_relative_error_vs_frequency`
- Можно сохранить артефакты и использовать обученную модель как `Preconditioner` внутри solver workflow


In [1]:
from __future__ import annotations

import csv
import json
import os
import random
import sys
import tempfile
from dataclasses import asdict
from pathlib import Path

MPLCONFIGDIR = Path(tempfile.gettempdir()) / "auto_preconditioner_mpl"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))
os.environ.setdefault("XDG_CACHE_HOME", str(MPLCONFIGDIR))

import matplotlib.pyplot as plt
import numpy as np
from scipy.sparse.linalg import LinearOperator

SEED = 0
np.random.seed(SEED)
random.seed(SEED)

ROOT = Path.cwd().resolve()
if not (ROOT / "preconditioner").exists():
    ROOT = ROOT.parent

PRECONDITIONER_SRC = ROOT / "preconditioner"
TESTS_DIR = ROOT / "tests"
EXPERIMENTS_DIR = ROOT / "experiments"

for path in (PRECONDITIONER_SRC, TESTS_DIR, EXPERIMENTS_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from benchmark_real_data import DATASET_ALIASES, build_impedance_matrix, build_rhs, load_compressed_dataset, resolve_data_dir, run_real_data_benchmark
from cnn import ConvolutionalInverseApproximator, evaluate_cnn_inverse_model, torch_is_available
from cnn_real_data_demo import DemoRecord, consistency_error, jacobi_inverse, relative_frobenius_error, run_demo, uniform_train_test_split
from solvers import solve_gmres

try:
    import pandas as pd
except ImportError:
    pd = None

if not torch_is_available():
    raise RuntimeError("PyTorch is required for this notebook. Install it with `pip install torch`.")


def preview_rows(rows: list[dict]):
    if pd is not None:
        return pd.DataFrame(rows)
    return rows


def write_rows_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


print({
    "root": ROOT.as_posix(),
    "datasets": sorted(DATASET_ALIASES),
    "torch_available": torch_is_available(),
    "pandas_available": pd is not None,
    "mpl_config_dir": str(MPLCONFIGDIR),
})


{'root': '/Users/albert/Research/auto-preconditioner', 'datasets': ['large', 'small'], 'torch_available': True, 'pandas_available': False, 'mpl_config_dir': '/var/folders/qf/xy929rcx1xnc19y51wt8zct00000gn/T/auto_preconditioner_mpl'}


## Step 1 - Pick a configuration

In [2]:
FAST_MODE = True
DATASET_NAME = "large"
RES_MIN = 38e6
RES_MAX = 42e6
TOL = 1e-6
MAXITER = 200
RESTART = 40

if FAST_MODE:
    FREQ_LIMIT = 12
    TRAIN_COUNT = 8
    TEST_COUNT = 4
    HIDDEN_CHANNELS = 10
    BASIS_RANK = 4
    EPOCHS = 4
else:
    FREQ_LIMIT = 24
    TRAIN_COUNT = 16
    TEST_COUNT = 8
    HIDDEN_CHANNELS = 12
    BASIS_RANK = 6
    EPOCHS = 10

BATCH_SIZE = 2
LEARNING_RATE = 2e-3
CONSISTENCY_WEIGHT = 0.3
OUTPUT_DIR = ROOT / "experiments" / (
    "cnn_real_data_demo_output_notebook_fast" if FAST_MODE else "cnn_real_data_demo_output_notebook"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "fast_mode": FAST_MODE,
    "dataset": DATASET_NAME,
    "res_min_hz": RES_MIN,
    "res_max_hz": RES_MAX,
    "freq_limit": FREQ_LIMIT,
    "train_count": TRAIN_COUNT,
    "test_count": TEST_COUNT,
    "hidden_channels": HIDDEN_CHANNELS,
    "basis_rank": BASIS_RANK,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "consistency_weight": CONSISTENCY_WEIGHT,
    "output_dir": OUTPUT_DIR.as_posix(),
}
config


{'fast_mode': True,
 'dataset': 'large',
 'res_min_hz': 38000000.0,
 'res_max_hz': 42000000.0,
 'freq_limit': 12,
 'train_count': 8,
 'test_count': 4,
 'hidden_channels': 10,
 'basis_rank': 4,
 'epochs': 4,
 'batch_size': 2,
 'learning_rate': 0.002,
 'consistency_weight': 0.3,
 'output_dir': '/Users/albert/Research/auto-preconditioner/experiments/cnn_real_data_demo_output_notebook_fast'}

## Step 2 - Load the real dataset

In [3]:
data_path = resolve_data_dir(dataset=DATASET_NAME, data_dir=None)
dataset = load_compressed_dataset(data_path)
freqs = dataset.selected_freqs(res_min=RES_MIN, res_max=RES_MAX, limit=FREQ_LIMIT)
train_freqs, test_freqs = uniform_train_test_split(freqs, TRAIN_COUNT, TEST_COUNT)

split_summary = {
    "data_dir": dataset.data_dir.as_posix(),
    "matrix_size": dataset.n,
    "available_freqs_in_window": int(len(freqs)),
    "train_freqs": int(len(train_freqs)),
    "test_freqs": int(len(test_freqs)),
    "first_freq_mhz": float(freqs[0] / 1e6),
    "last_freq_mhz": float(freqs[-1] / 1e6),
}
split_summary


{'data_dir': '/Users/albert/Research/auto-preconditioner/tests/data/compressed_data555',
 'matrix_size': 450,
 'available_freqs_in_window': 12,
 'train_freqs': 8,
 'test_freqs': 4,
 'first_freq_mhz': 38.00467995329644,
 'last_freq_mhz': 38.09223916986073}

In [4]:
freqs_mhz = freqs / 1e6
train_freqs_mhz = train_freqs / 1e6
test_freqs_mhz = test_freqs / 1e6

plt.figure(figsize=(10, 3.2))
plt.plot(freqs_mhz, np.zeros_like(freqs_mhz), marker="o", linestyle="", alpha=0.35, label="all selected frequencies")
plt.plot(train_freqs_mhz, np.ones_like(train_freqs_mhz), marker="s", linestyle="", label="train")
plt.plot(test_freqs_mhz, -np.ones_like(test_freqs_mhz), marker="^", linestyle="", label="test")
plt.yticks([-1, 0, 1], ["test", "all", "train"])
plt.xlabel("Frequency, MHz")
plt.title("Uniform train/test split over the selected frequency window")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


/var/folders/qf/xy929rcx1xnc19y51wt8zct00000gn/T/ipykernel_12744/722975283.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 3 - Build the training targets
Для обучения CNN здесь в качестве target используется точная плотная inverse `A^{-1}` для каждой train frequency

In [18]:
train_matrices = [build_impedance_matrix(dataset, float(freq)) for freq in train_freqs]
train_targets = [np.linalg.inv(A) for A in train_matrices]
test_matrices = [build_impedance_matrix(dataset, float(freq)) for freq in test_freqs]
test_targets = [np.linalg.inv(A) for A in test_matrices]
{
    "n_train_matrices": len(train_matrices),
    "matrix_shape": train_matrices[0].shape,
    "train_target_fro_norm_mean": float(np.mean([np.linalg.norm(T, ord="fro") for T in train_targets])),
}


{'n_train_matrices': 8,
 'matrix_shape': (450, 450),
 'train_target_fro_norm_mean': 2506.000426781673}

## Step 4 - Train `ConvolutionalInverseApproximator`

In [19]:
model = ConvolutionalInverseApproximator(
    matrix_size=dataset.n,
    hidden_channels=HIDDEN_CHANNELS,
    basis_rank=BASIS_RANK,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    consistency_weight=CONSISTENCY_WEIGHT,
    seed=SEED,
)
model.fit(train_matrices, train_targets, verbose=True)
{
    "architecture": model.describe_architecture(),
    "best_epoch": model.history.best_epoch + 1 if model.history is not None else None,
}

[CNNApprox] epoch   1/4 train=1.62038e+00 val=8.03298e-01
[CNNApprox] epoch   4/4 train=1.45910e+00 val=6.45458e-01


{'architecture': 'Jacobi + low-rank CNN correction: hidden_channels=10, dilations=(1, 2, 4, 2, 1), basis_rank=4, aux=row/column coordinates, diagonal mask, Jacobi residual channels, consistency_weight=0.300',
 'best_epoch': 4}

## Step 5 - Plot training history

In [9]:
history = model.history
assert history is not None

epochs_axis = np.arange(1, len(history.train_loss) + 1)
plt.figure(figsize=(10, 4))
plt.plot(epochs_axis, history.train_loss, label="train total", linewidth=2)
plt.plot(epochs_axis, history.val_loss, label="val total", linewidth=2)
plt.plot(epochs_axis, history.train_target_loss, label="train target", linestyle="--")
plt.plot(epochs_axis, history.train_consistency_loss, label="train consistency", linestyle=":")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNNApprox training history")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_loss.png", dpi=180)
plt.show()


/var/folders/qf/xy929rcx1xnc19y51wt8zct00000gn/T/ipykernel_12744/193946209.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 6 - Evaluate on held-out frequencies
Для каждой test frequency считаются:
- число итераций GMRES;
- относительный residual solver-а;
- consistency `||PA - I||_F / n`;
- относительная ошибка inverse-оператора.

In [10]:
records: list[DemoRecord] = []
for freq in test_freqs:
    A = build_impedance_matrix(dataset, float(freq))
    rhs = build_rhs(dataset, float(freq))
    target = np.linalg.inv(A)
    P_diag = jacobi_inverse(A)
    P_cnn = model.predict_inverse(A)

    none_result = solve_gmres(A, rhs, None, TOL, MAXITER, restart=RESTART)
    diag_result = solve_gmres(
        A,
        rhs,
        LinearOperator(A.shape, matvec=lambda x, P=P_diag: P @ x, dtype=A.dtype),
        TOL,
        MAXITER,
        restart=RESTART,
    )
    cnn_result = solve_gmres(
        A,
        rhs,
        LinearOperator(A.shape, matvec=lambda x, P=P_cnn: P @ x, dtype=A.dtype),
        TOL,
        MAXITER,
        restart=RESTART,
    )

    records.append(
        DemoRecord(
            frequency_hz=float(freq),
            gmres_none_iter=none_result.n_iter,
            gmres_diag_iter=diag_result.n_iter,
            gmres_cnn_iter=cnn_result.n_iter,
            gmres_none_residual=float(np.linalg.norm(rhs - A @ none_result.x) / np.linalg.norm(rhs)),
            gmres_diag_residual=float(np.linalg.norm(rhs - A @ diag_result.x) / np.linalg.norm(rhs)),
            gmres_cnn_residual=float(np.linalg.norm(rhs - A @ cnn_result.x) / np.linalg.norm(rhs)),
            diag_consistency=consistency_error(P_diag, A),
            cnn_consistency=consistency_error(P_cnn, A),
            cnn_relative_inverse_error=relative_frobenius_error(P_cnn, target),
        )
    )

record_rows = [asdict(record) for record in records]
preview_rows(record_rows)


[{'frequency_hz': 38020599.81085358,
  'gmres_none_iter': 47,
  'gmres_diag_iter': 47,
  'gmres_cnn_iter': 6,
  'gmres_none_residual': 4.352803992132602e-07,
  'gmres_diag_residual': 4.29601328638074e-07,
  'gmres_cnn_residual': 2.5966992745151518e-08,
  'diag_consistency': 0.07059392785878296,
  'cnn_consistency': 0.004065477784149962,
  'cnn_relative_inverse_error': 0.264224088470753},
 {'frequency_hz': 38044479.5971893,
  'gmres_none_iter': 47,
  'gmres_diag_iter': 47,
  'gmres_cnn_iter': 4,
  'gmres_none_residual': 6.374758989873191e-07,
  'gmres_diag_residual': 6.512538668457772e-07,
  'gmres_cnn_residual': 6.560880106660427e-08,
  'diag_consistency': 0.07164282739373729,
  'cnn_consistency': 0.001504470036719988,
  'cnn_relative_inverse_error': 0.11478237070198821},
 {'frequency_hz': 38068359.383525014,
  'gmres_none_iter': 47,
  'gmres_diag_iter': 47,
  'gmres_cnn_iter': 5,
  'gmres_none_residual': 8.851509487461479e-07,
  'gmres_diag_residual': 8.91714013218815e-07,
  'gmres_cn

In [11]:
test_rel_error_mean, test_rel_error_std, test_kappa_mean, test_kappa_std = evaluate_cnn_inverse_model(
    model,
    test_matrices,
    test_targets,
)

summary = {
    "dataset": dataset.data_dir.as_posix(),
    "matrix_size": dataset.n,
    "train_frequencies_hz": [float(freq) for freq in train_freqs],
    "test_frequencies_hz": [float(freq) for freq in test_freqs],
    "architecture": model.describe_architecture(),
    "gmres_none_mean_iter": float(np.mean([record.gmres_none_iter for record in records])),
    "gmres_diag_mean_iter": float(np.mean([record.gmres_diag_iter for record in records])),
    "gmres_cnn_mean_iter": float(np.mean([record.gmres_cnn_iter for record in records])),
    "diag_consistency_mean": float(np.mean([record.diag_consistency for record in records])),
    "cnn_consistency_mean": float(np.mean([record.cnn_consistency for record in records])),
    "cnn_relative_inverse_error_mean": float(np.mean([record.cnn_relative_inverse_error for record in records])),
    "eval_rel_error_mean": float(test_rel_error_mean),
    "eval_rel_error_std": float(test_rel_error_std),
    "eval_kappa_mean": float(test_kappa_mean),
    "eval_kappa_std": float(test_kappa_std),
}
summary


{'dataset': '/Users/albert/Research/auto-preconditioner/tests/data/compressed_data555',
 'matrix_size': 450,
 'train_frequencies_hz': [38004679.95329644,
  38012639.882075004,
  38028559.73963215,
  38036519.668410726,
  38052439.52596787,
  38060399.45474645,
  38076319.31230359,
  38092239.169860736],
 'test_frequencies_hz': [38020599.81085358,
  38044479.5971893,
  38068359.383525014,
  38084279.24108216],
 'architecture': 'Jacobi + low-rank CNN correction: hidden_channels=10, dilations=(1, 2, 4, 2, 1), basis_rank=4, aux=row/column coordinates, diagonal mask, Jacobi residual channels, consistency_weight=0.300',
 'gmres_none_mean_iter': 47.25,
 'gmres_diag_mean_iter': 47.25,
 'gmres_cnn_mean_iter': 5.5,
 'diag_consistency_mean': 0.07210382807330114,
 'cnn_consistency_mean': 0.0038849359811329457,
 'cnn_relative_inverse_error_mean': 0.24933565599915697,
 'eval_rel_error_mean': 0.24933565599915697,
 'eval_rel_error_std': 0.09264446171913167,
 'eval_kappa_mean': 1.4465007796024907,
 'ev

## Step 7 - Rebuild the demo plots
- `iterations_vs_frequency`: помогает ли CNN solver-у быстрее сходиться;
- `consistency_vs_frequency`: насколько `P @ A` близок к единичной матрице;
- `inverse_relative_error`: насколько хорошо предсказан сам inverse.

In [12]:
freqs_mhz_eval = np.asarray([record.frequency_hz for record in records], dtype=np.float64) / 1e6
none_iter = np.asarray([record.gmres_none_iter for record in records], dtype=np.float64)
diag_iter = np.asarray([record.gmres_diag_iter for record in records], dtype=np.float64)
cnn_iter = np.asarray([record.gmres_cnn_iter for record in records], dtype=np.float64)

plt.figure(figsize=(10, 4))
plt.plot(freqs_mhz_eval, none_iter, marker="o", label="None", linewidth=1.8)
plt.plot(freqs_mhz_eval, diag_iter, marker="s", label="Diagonal", linewidth=1.8)
plt.plot(freqs_mhz_eval, cnn_iter, marker="^", label="CNNApprox", linewidth=2.2)
plt.xlabel("Frequency, MHz")
plt.ylabel("GMRES iterations")
plt.title("GMRES iterations on held-out frequencies")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "gmres_iterations_vs_frequency.png", dpi=180)
plt.show()


/var/folders/qf/xy929rcx1xnc19y51wt8zct00000gn/T/ipykernel_12744/2574001457.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
diag_cons = np.asarray([record.diag_consistency for record in records], dtype=np.float64)
cnn_cons = np.asarray([record.cnn_consistency for record in records], dtype=np.float64)

plt.figure(figsize=(10, 4))
plt.semilogy(freqs_mhz_eval, diag_cons, marker="s", label="Diagonal", linewidth=1.8)
plt.semilogy(freqs_mhz_eval, cnn_cons, marker="^", label="CNNApprox", linewidth=2.2)
plt.xlabel("Frequency, MHz")
plt.ylabel(r"$||PA - I||_F / n$")
plt.title("Preconditioner consistency")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "preconditioner_consistency_vs_frequency.png", dpi=180)
plt.show()


/var/folders/qf/xy929rcx1xnc19y51wt8zct00000gn/T/ipykernel_12744/1418891061.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
cnn_inverse_error = np.asarray([record.cnn_relative_inverse_error for record in records], dtype=np.float64)

plt.figure(figsize=(10, 4))
plt.semilogy(freqs_mhz_eval, cnn_inverse_error, marker="^", linewidth=2.2)
plt.xlabel("Frequency, MHz")
plt.ylabel("Relative Frobenius error")
plt.title("CNN inverse approximation error")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "inverse_relative_error_vs_frequency.png", dpi=180)
plt.show()


/var/folders/qf/xy929rcx1xnc19y51wt8zct00000gn/T/ipykernel_12744/2492569615.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 8 - Save artifacts and reuse the trained model

In [15]:
write_rows_csv(OUTPUT_DIR / "records.csv", record_rows)
with (OUTPUT_DIR / "summary.json").open("w") as f:
    json.dump(summary, f, indent=2)

example_preconditioner = model.as_preconditioner(test_matrices[0], name="CNNApprox")
example_linear_operator = example_preconditioner.as_linear_operator()

{
    "saved_records": (OUTPUT_DIR / "records.csv").as_posix(),
    "saved_summary": (OUTPUT_DIR / "summary.json").as_posix(),
    "linear_operator_shape": example_linear_operator.shape,
}


{'saved_records': '/Users/albert/Research/auto-preconditioner/experiments/cnn_real_data_demo_output_notebook_fast/records.csv',
 'saved_summary': '/Users/albert/Research/auto-preconditioner/experiments/cnn_real_data_demo_output_notebook_fast/summary.json',
 'linear_operator_shape': (450, 450)}

## Step 9 - Optional full benchmark across all solvers

In [16]:
RUN_FULL_BENCHMARK = False

if RUN_FULL_BENCHMARK:
    benchmark_summary, benchmark_records, skipped = run_real_data_benchmark(
        data_dir=data_path,
        res_min=RES_MIN,
        res_max=RES_MAX,
        freq_limit=FREQ_LIMIT,
        tol=TOL,
        maxiter=MAXITER,
        restart=RESTART,
        include_ridge=False,
        include_cnn=True,
        cnn_train_limit=TRAIN_COUNT,
        cnn_hidden_channels=HIDDEN_CHANNELS,
        cnn_basis_rank=BASIS_RANK,
        cnn_epochs=EPOCHS,
        cnn_batch_size=BATCH_SIZE,
        cnn_learning_rate=LEARNING_RATE,
        cnn_consistency_weight=CONSISTENCY_WEIGHT,
        verbose=True,
    )
    preview_rows([asdict(row) for row in benchmark_summary])
else:
    print("Set RUN_FULL_BENCHMARK = True to execute the full solver benchmark.")


Set RUN_FULL_BENCHMARK = True to execute the full solver benchmark.


In [17]:
# Exercise answer scaffold
ALTERNATIVE_BASIS_RANK = BASIS_RANK + 2
RUN_ALTERNATIVE = False

if RUN_ALTERNATIVE:
    alt_records, alt_output_dir = run_demo(
        dataset=DATASET_NAME,
        res_min=RES_MIN,
        res_max=RES_MAX,
        freq_limit=FREQ_LIMIT,
        train_count=TRAIN_COUNT,
        test_count=TEST_COUNT,
        hidden_channels=HIDDEN_CHANNELS,
        basis_rank=ALTERNATIVE_BASIS_RANK,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        consistency_weight=CONSISTENCY_WEIGHT,
        output_dir=OUTPUT_DIR / f"basis_rank_{ALTERNATIVE_BASIS_RANK}",
    )
    print({
        "alternative_basis_rank": ALTERNATIVE_BASIS_RANK,
        "alternative_output_dir": alt_output_dir.as_posix(),
        "alternative_gmres_cnn_mean_iter": float(np.mean([row.gmres_cnn_iter for row in alt_records])),
    })
else:
    print("Set RUN_ALTERNATIVE = True to launch an additional comparison run.")


Set RUN_ALTERNATIVE = True to launch an additional comparison run.
